In [0]:
# %sql

# CREATE SCHEMA IF NOT EXISTS ECOMMERCE.SILVER;

In [0]:
%sql
SELECT * FROM ecommerce.bronze.customers limit 10;

## Loading Libraries for future usage

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number, col


# Loading data from Bronze layer

## Business Rule 1: Drop Duplicates 

We load the data from the bronze layer table and we drop duplicates (requirement as a part of customer business rule 1- drop duplicates )

In [0]:
customer_df = spark.read.table("ecommerce.bronze.customers").dropDuplicates()

## Business rule 2: Keep the latest record where appropriate.

In [0]:
window_spec = Window.partitionBy("customer_id").orderBy(col("updated_at").desc())
customer_df = customer_df.withColumn("rn", row_number().over(window_spec)).filter(col("rn") == 1).drop("rn")

## Business rule 3: Trim whitespace.

In [0]:
string_columns = [
    "first_name",
    "last_name",
    "email",
    "phone",
    "city",
    "state",
    "country",
    "customer_status",
]

for c in string_columns:
    customer_df = customer_df.withColumn(c, trim(col(c)))

## Business Rule 4: Convert email to lowercase.

In [0]:
customer_df = customer_df.withColumn("email", lower(col("email")))

## Business Rule 5: Validate Email format 

In [0]:
customer_df = customer_df.withColumn(
    "email_valid",
    when(
        col("email").rlike(r"^[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}$"), True
    ).otherwise(False),
)

## Business Rule 6: Validate missing email or phone

In [0]:
customer_df = customer_df.withColumn(
    "email", when(trim(col("email")) == "", None).otherwise(col("email"))
).withColumn("phone", when(trim(col("phone")) == "", None).otherwise(col("phone")))

## Business rule 7: Standardise city

In [0]:
customer_df = customer_df.withColumn("city", initcap(trim(col("city")))).withColumn(
    "state", initcap(trim(col("state")))
)

## Business rule 8: Validate customer status 

In [0]:
customer_df = customer_df.withColumn(
    "customer_status_valid",
    when(
        col("customer_status").isin("ACTIVE", "INACTIVE", "SUSPENDED"), True
    ).otherwise(False),
)

In [0]:
display(customer_df)

## Writing results in sql table 

In [0]:
customer_df.write.format("delta").mode("append").saveAsTable(
    "ecommerce.silver.customers"
)

In [0]:
%sql

SELECT * FROM ECOMMERCE.SILVER.CUSTOMERS LIMIT 100;